In [ ]:
import sys
sys.path.append('./mesh')
import os
import yaml
import numpy as np
from mesh.template import Template
from mesh.model import MeshDeformer
from mesh.render import MultiViewRenderer
from mesh.losses import *
from dataloader import MultiViewDataset
from utils import load_cameras, convert_cameras
from train_mesh import MeshTrainer

In [ ]:
# Load config 
config_path = './config/default.yaml'
with open(config_path, "r") as file:
        config = yaml.safe_load(file)

In [ ]:
# Load Cameras
dataset_path = config['dataset']
cam_path = os.path.join(dataset_path, 'cameras.npy')
wh = eval(config['image_size']) # wh
hw = (wh[1], wh[0])
world_scale = config['world_scale']

cameras = load_cameras(cam_path)
whs = np.array(wh)[None, ...].repeat(cameras.nc, axis=0)
cameras.scale(new_sizes=whs, world_scale=world_scale)
cameras = convert_cameras(cameras)

In [ ]:
# Load Template 
template_path = os.path.join(dataset_path, 'template.pth')
device = config['device']
batch_size = config['batch_size']
template = Template(template_path, device=device)

In [ ]:
# Construct Trainer 
renderer = MultiViewRenderer(image_size=hw, cameras=cameras, lights=None, device=device, faces_per_pixel=5, sigma=1e-6)
model = MeshDeformer(batch_size=batch_size, template=template, renderer=renderer, device=device)
dataset = MultiViewDataset(root_path=dataset_path, resize=hw, mask_invert=False, world_scale=world_scale)
trainer = MeshTrainer(model=model, dataset=dataset)

In [ ]:
# Mesh fitting 
trainer.train(config)